In [1]:
import numpy as np
import pandas as pd

In [2]:
df = pd.read_csv("IMDB Dataset.csv")

In [3]:
df.shape

(50000, 2)

In [4]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [5]:
df.drop_duplicates(inplace=True)

In [6]:
df.shape

(49582, 2)

# Pre processing

## 1. convert to lowercase

In [7]:
df["review"] = df["review"].str.lower()

## 2. remove URL's

In [8]:
import re
#re stands for Regular Expression. -> It is a Python library used to search, find, replace, or remove patterns in text

In [9]:
def remove_url(text):
    text = str(text)
    text = re.sub(r"http\S+", "", text) #(pattern, replacement, string)->\S+ Any character that is NOT whitespace
    return text
df["review"] = df["review"].apply(remove_url)

## 3. remove puntuation

In [10]:
def remove_punc(text):
    text = re.sub(r"[^A-Za-z0-9\s+]", "", text) #(pattern, replacement, string)
    return text
df["review"] = df["review"].apply(remove_punc)

## 4. removing html tags

In [11]:
def remove_tags(text):
    text = re.sub(r"<.*?>", "", text) #(pattern, replacement, string)
    return text
df["review"] = df["review"].apply(remove_tags)

## 5. removing stopwords

In [12]:
import nltk # NLTK is a Python library for NLP that provides tools and resources for processing, analyzing, and understanding human language text.

In [13]:
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Debangana\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Debangana\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Debangana\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [14]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [15]:
sample = "I am going to be a millionaire very soon"
tokens = word_tokenize(sample)
tokens

['I', 'am', 'going', 'to', 'be', 'a', 'millionaire', 'very', 'soon']

In [16]:
def remove_stopwords(text):
    tokens = word_tokenize(text)
    stop_words = stopwords.words("english")

    for word in tokens:
        if word in stop_words:
            text = text.replace(word, "") 
    return text
df["review"] = df["review"].apply(remove_stopwords)

In [17]:
df.head()

,review,sentiment
0,e revewers nted wtchg 1 oz epode ll ho...,positive
1,wderful ltle producti br br filming techniqu...,positive
2,thought ths wderful wy spend tme o hot s...,positive
3,bsclly res fmly lttle boy jke thks res zom...,negative
4,petter mtte love time mey vully stunng fi...,positive


## 6. Stemming

In [18]:
#running -> run
#played -> play
#PorterStemming

from nltk.stem import PorterStemmer

In [19]:
def stemming(text):
    ps = PorterStemmer()
    stemmed_words = []
    
    tokens = word_tokenize(text)
    
    for word in tokens:
        stemmed_tokens = ps.stem(word)
        stemmed_words.append(stemmed_tokens)

    return " ".join(stemmed_words) #combines all list elements into a single string.

df["review"] = df["review"].apply(stemming)

In [20]:
df.head()

,review,sentiment
0,e revew nted wtchg 1 oz epod ll hook y rght ex...,positive
1,wder ltle producti br br film techniqu unssum ...,positive
2,thought th wder wy spend tme o hot summer week...,positive
3,bsclli re fmli lttle boy jke thk re zomb close...,negative
4,petter mtte love time mey vulli stunng film wt...,positive


## 7. Encoding

In [21]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df["sentiment"] = le.fit_transform(df["sentiment"])

In [22]:
y = df["sentiment"]

In [23]:
y

0        1
1        1
2        1
3        0
4        1
        ..
49995    1
49996    0
49997    0
49998    0
49999    0
Name: sentiment, Length: 49582, dtype: int32

## 8. Vectorization

In [24]:
from sklearn.feature_extraction.text import TfidfVectorizer

tf = TfidfVectorizer(max_features=5000)

X = tf.fit_transform(df["review"])

In [25]:
X

<49582x5000 sparse matrix of type '<class 'numpy.float64'>'
	with 4057448 stored elements in Compressed Sparse Row format>

## DataSet and DataLoader

In [26]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [27]:
X_train.shape

(39665, 5000)

In [28]:
X_test.shape

(9917, 5000)

In [29]:
X_train

<39665x5000 sparse matrix of type '<class 'numpy.float64'>'
	with 3248551 stored elements in Compressed Sparse Row format>

In [30]:
X_train = X_train.toarray()
X_test = X_test.toarray()

In [31]:
X_train

array([[0.21333721, 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       ...,
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ]])

In [32]:
import torch
from torch.utils.data import TensorDataset, DataLoader

In [33]:
train_set = TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train.values).float() 
)

test_set = TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test.values).float() 
)

In [34]:
train_loader = DataLoader(train_set, shuffle=True, batch_size=64)
test_loader = DataLoader(test_set, batch_size=64)

## Build our RNN

In [35]:
import torch.nn as nn
import torch.optim as optimizer

In [36]:
class RNN(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=1):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        #RNN 
        self.rnn = nn.RNN(input_size, hidden_size=128, num_layers=1, batch_first=True) 
        #Expects input in the order (batch_size, sequence_length, input_size)

        #fc
        self.fc = nn.Linear(hidden_size, 1) #i/p, o/p

    def forward(self, x):
        # optional step => shape(numof layers, batch_size, hidden_size)
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)
        # Creates the initial hidden state filled with zeros.
        
        out, _ = self.rnn(x, h0)
        #1st val = hidden state of all the timesteps => batch, seq_len, hidden size
        # 2nd val = final hidden state of last timestep

        out = self.fc(out[:, -1, :]) 
        # : → Select all samples in the batch.
        # -1 → Select the last timestep.
        # : → Select all 128 hidden features.
        return out

In [37]:
input_size = X_train.shape[1]

model = RNN(input_size)

crie = nn.BCELoss()
optimz= optimizer.Adam(model.parameters())

## Training model

In [38]:
epochs = 10
for epoch in range(epochs):
    model.train()

    for xb, yb in train_loader:
        optimz.zero_grad()

        xb = xb.unsqueeze(1) # tf-idf ussualy gives 2D matrix, but we want 3D

        outputs = model(xb)

        outputs = torch.sigmoid(outputs).squeeze() # covt to prob

        loss = crie(outputs, yb)
        loss.backward()
        optimz.step()

    print(f"epoch = {epoch+1}/{epochs} and loss = {loss.item()}")

epoch = 1/10 and loss = 0.2789967954158783
epoch = 2/10 and loss = 0.26165011525154114
epoch = 3/10 and loss = 0.2336806058883667
epoch = 4/10 and loss = 0.25977373123168945
epoch = 5/10 and loss = 0.20330239832401276
epoch = 6/10 and loss = 0.35473164916038513
epoch = 7/10 and loss = 0.29987186193466187
epoch = 8/10 and loss = 0.2606016993522644
epoch = 9/10 and loss = 0.2943263649940491
epoch = 10/10 and loss = 0.1611623615026474


## Evaluation

In [39]:
model.eval()

with torch.no_grad():
    correct_vals, tot_vals = 0,0
    for xb, yb in test_loader:
        xb = xb.unsqueeze(1) # tf-idf ussualy gives 2D matrix, but we want 3D

        outputs = model(xb)

        predicted = (torch.sigmoid(outputs.squeeze())>0.5).float()

        tot_vals += yb.size(0)
        correct_vals += (predicted == yb).sum().item()

    print(f"accuracy = {correct_vals/tot_vals*100}")

accuracy = 85.72148835333266


In [42]:
#85%